# Message Passing

> **Source:** `repo1/agent_communication.py` → `demo_message_passing()`


## Imports


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from typing_extensions import TypedDict, Annotated
from typing import Literal
from pydantic import BaseModel, Field
import operator
import json
from dotenv import load_dotenv


## Configuration & Setup


In [ ]:
load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## Class: `MessagePassingState`


In [ ]:
class MessagePassingState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    current_phase: str


## Class: `SharedFieldsState`


In [ ]:
class SharedFieldsState(TypedDict):
    query: str
    # Each agent writes to its own field — others can read it
    raw_data: Annotated[list[dict], operator.add]
    analysis: str
    recommendations: list[str]
    confidence_score: float


## Class: `BlackboardState`


In [ ]:
class BlackboardState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # Blackboard fields — the shared workspace
    topic: str
    drafts: Annotated[list[str], operator.add]
    critiques: Annotated[list[str], operator.add]
    iteration: int
    is_approved: bool


## Helper Function: `create_message_passing_pipeline`


In [ ]:
def create_message_passing_pipeline():
    """Agents communicate by appending messages that others can read."""

    def researcher(state: MessagePassingState) -> dict:
        """Researches the topic and posts findings as a message."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a researcher. Read the user's question, "
                        "research it, and post your findings. Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[RESEARCHER]: {response.content}", name="researcher"
                )
            ],
            "current_phase": "fact_checker",
        }

    def fact_checker(state: MessagePassingState) -> dict:
        """Reads the researcher's message and validates the claims."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a fact-checker. Read the researcher's findings "
                        "in the conversation and validate or challenge them. "
                        "Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[FACT-CHECKER]: {response.content}", name="fact_checker"
                )
            ],
            "current_phase": "summarizer",
        }

    def summarizer(state: MessagePassingState) -> dict:
        """Reads all previous messages and creates a final summary."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a summarizer. Read the researcher's findings and "
                        "the fact-checker's review. Produce a final, accurate summary. "
                        "Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(content=f"[SUMMARY]: {response.content}", name="summarizer")
            ],
            "current_phase": "done",
        }

    graph = StateGraph(MessagePassingState)

    graph.add_node("researcher", researcher)
    graph.add_node("fact_checker", fact_checker)
    graph.add_node("summarizer", summarizer)

    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "fact_checker")
    graph.add_edge("fact_checker", "summarizer")
    graph.add_edge("summarizer", END)

    return graph.compile()


## Helper Function: `create_shared_fields_pipeline`


In [ ]:
def create_shared_fields_pipeline():
    """Agents communicate through typed state fields, not messages."""

    def data_collector(state: SharedFieldsState) -> dict:
        """Collects data and writes to the raw_data field."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a data collector. Given the query, produce 3 data points "
                        "as a JSON array of objects with 'source' and 'finding' keys. "
                        "Return ONLY the JSON array, no markdown."
                    )
                ),
                HumanMessage(content=state["query"]),
            ]
        )

        try:
            data = json.loads(response.content)
        except json.JSONDecodeError:
            data = [{"source": "llm", "finding": response.content}]

        return {"raw_data": data}

    def analyst(state: SharedFieldsState) -> dict:
        """Reads raw_data field, writes analysis and confidence."""
        data_summary = json.dumps(state["raw_data"], indent=2)

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a data analyst. Analyze the collected data and provide: "
                        "1) A brief analysis (2-3 sentences), and "
                        "2) A confidence score from 0.0 to 1.0. "
                        "Format: ANALYSIS: <text>\nCONFIDENCE: <number>"
                    )
                ),
                HumanMessage(
                    content=f"Query: {state['query']}\n\nData:\n{data_summary}"
                ),
            ]
        )

        content = response.content
        analysis = content
        confidence = 0.7  # default

        if "CONFIDENCE:" in content:
            parts = content.split("CONFIDENCE:")
            analysis = parts[0].replace("ANALYSIS:", "").strip()
            try:
                confidence = float(parts[1].strip())
            except ValueError:
                confidence = 0.7

        return {"analysis": analysis, "confidence_score": confidence}

    def advisor(state: SharedFieldsState) -> dict:
        """Reads analysis + confidence, writes recommendations."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a strategic advisor. Based on the analysis and "
                        "confidence score, provide 3 actionable recommendations. "
                        "Return them as a JSON array of strings. "
                        "Return ONLY the JSON array, no markdown."
                    )
                ),
                HumanMessage(
                    content=(
                        f"Query: {state['query']}\n"
                        f"Analysis: {state['analysis']}\n"
                        f"Confidence: {state['confidence_score']}"
                    )
                ),
            ]
        )

        try:
            recs = json.loads(response.content)
        except json.JSONDecodeError:
            recs = [response.content]

        return {"recommendations": recs}

    graph = StateGraph(SharedFieldsState)

    graph.add_node("data_collector", data_collector)
    graph.add_node("analyst", analyst)
    graph.add_node("advisor", advisor)

    graph.add_edge(START, "data_collector")
    graph.add_edge("data_collector", "analyst")
    graph.add_edge("analyst", "advisor")
    graph.add_edge("advisor", END)

    return graph.compile()


## Helper Function: `create_blackboard_system`


In [ ]:
def create_blackboard_system():
    """
    Blackboard pattern: multiple agents read/write a shared workspace.
    A drafter writes, a critic reviews, and they iterate until approved.
    """

    class ApprovalDecision(BaseModel):
        approved: bool = Field(description="Whether the draft is good enough")
        feedback: str = Field(description="Specific feedback if not approved")

    critic_llm = llm.with_structured_output(ApprovalDecision)

    def drafter(state: BlackboardState) -> dict:
        """Reads critiques from blackboard, writes improved draft."""
        context_parts = [f"Topic: {state['topic']}"]

        if state["drafts"]:
            context_parts.append(f"Previous draft: {state['drafts'][-1]}")
        if state["critiques"]:
            context_parts.append(f"Feedback to address: {state['critiques'][-1]}")

        context = "\n".join(context_parts)

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a skilled writer. Write or revise a short paragraph "
                        "(3-4 sentences) based on the topic and any feedback provided. "
                        "If there's feedback, directly address it in your revision."
                    )
                ),
                HumanMessage(content=context),
            ]
        )

        return {
            "drafts": [response.content],
            "messages": [
                AIMessage(
                    content=f"[DRAFTER iteration {state['iteration'] + 1}]: {response.content}",
                    name="drafter",
                )
            ],
            "iteration": state["iteration"] + 1,
        }

    def critic(state: BlackboardState) -> dict:
        """Reads latest draft from blackboard, writes critique or approves."""
        latest_draft = state["drafts"][-1] if state["drafts"] else "No draft yet"

        decision = critic_llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a strict editor. Review the draft for clarity, accuracy, "
                        "and engagement. Approve ONLY if it's genuinely good. "
                        "If iteration is 3 or more, be more lenient."
                    )
                ),
                HumanMessage(
                    content=(
                        f"Topic: {state['topic']}\n"
                        f"Iteration: {state['iteration']}\n"
                        f"Draft: {latest_draft}"
                    )
                ),
            ]
        )

        # Force approval after 3 iterations to prevent infinite loops
        approved = decision.approved or state["iteration"] >= 3

        result = {
            "is_approved": approved,
            "messages": [
                AIMessage(
                    content=f"[CRITIC]: {'APPROVED' if approved else 'REVISION NEEDED'} - {decision.feedback}",
                    name="critic",
                )
            ],
        }

        if not approved:
            result["critiques"] = [decision.feedback]

        return result

    def route_after_critic(state: BlackboardState) -> Literal["drafter", "end"]:
        """Loop back to drafter if not approved."""
        if state["is_approved"]:
            return "end"
        return "drafter"

    graph = StateGraph(BlackboardState)

    graph.add_node("drafter", drafter)
    graph.add_node("critic", critic)

    graph.add_edge(START, "drafter")
    graph.add_edge("drafter", "critic")
    graph.add_conditional_edges(
        "critic", route_after_critic, {"drafter": "drafter", "end": END}
    )

    return graph.compile()


## Demo: Message Passing


In [ ]:
def demo_message_passing():
    """Demo message passing between agents."""
    agent = create_message_passing_pipeline()

    print("Message Passing Demo:\n")

    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content="What are the main benefits of renewable energy?")
            ],
            "current_phase": "researcher",
        }
    )

    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"{msg.content}\n")


## Execute


In [ ]:
demo_message_passing()
